## 📂 Querying Files (Lectura de Datos desde Volúmenes)

Hasta este momento hemos trabajado principalmente con **Delta Tables**, las cuales forman parte de la gobernanza de **Unity Catalog** y ofrecen funcionalidades avanzadas para la gestión de datos.

Sin embargo, dentro de Databricks Free Edition la gobernanza de la información se organiza mediante la siguiente jerarquía:

* 📂 **Catálogo (Catalog):** Es el nivel superior de Unity Catalog y permite organizar la información mediante diferentes esquemas.

* 📁 **Esquema (Schema):** Contenedor lógico donde se almacenan los distintos objetos gobernados por Unity Catalog, como Delta Tables, Volúmenes, Vistas, entre otros.

* 📦 **Volumen (Volume):** Puede entenderse como una carpeta gobernada donde almacenamos archivos en su estado más puro. Es decir, cualquier archivo que represente una fuente de datos (CSV, JSON, Parquet, Excel, etc.) puede almacenarse dentro de un volumen.

y/o

* 🗄️ **Delta Table:** Tabla gobernada por Unity Catalog que utiliza el formato Delta Lake, ofreciendo capacidades como transacciones ACID, historial de cambios y otras funcionalidades que hemos estudiado en los capítulos anteriores.

---

Hasta ahora hemos profundizado en el funcionamiento de las **Delta Tables** y en las características que las convierten en el estándar para almacenar información gobernada dentro de Databricks.

Los **Volúmenes**, por otro lado, representan la otra cara de la moneda.

Su propósito principal es almacenar archivos en su estado original dentro de Unity Catalog. A diferencia de las Delta Tables, estos archivos no adquieren automáticamente capacidades como Time Travel, Transaction Log o transacciones ACID. Si deseamos modificar su contenido, normalmente será necesario reemplazar el archivo completo.

---

### 🚀 ¿Qué veremos en este capítulo?

En este notebook nos enfocaremos únicamente en la **lectura de archivos almacenados en Volúmenes**, la cual puede realizarse mediante dos enfoques principales:

* ⚡ Spark SQL
* 🐍 PySpark (Siguiente capitulo)

Ambos permiten consultar la información almacenada en los Volúmenes, aunque la forma de lectura y las opciones disponibles dependerán del tipo de archivo que estemos utilizando.


### 🚀 Punto de Inicio en Databricks

Antes de trabajar con Delta Lake necesitamos una sesión de Spark activa.

Spark será el motor encargado de:

* ✅ Leer datos
* ✅ Transformarlos
* ✅ Procesarlos de forma distribuida
* ✅ Persistirlos como Delta Tables

In [0]:
from pyspark.sql import SparkSession # Puerta de entrada para trabajar con spark <-- SIEMPRE DEBEMOS IMPORTAR LA LLAVE MAESTRA QUE INICIA TODO.
from pyspark.sql.functions import *  # Funciones propias del módulo SQL de Spark, para trabajar sobre Dataframes.
spark = SparkSession.builder.appName("12QueryingFiles").getOrCreate() 
"""
^          ^__________^        ^_________^                               ^
|                |                   |                                   | 
Variable   Constructor de Sesión   Nombre Aplicación       Evita conflicto del SparkSession"""

print("🚀 Spark Session iniciada correctamente")

### ⚡ Spark SQL + Volúmenes

Una de las formas más sencillas de consultar archivos almacenados en un **Volumen** es utilizando **Spark SQL**. Para ello, disponemos de dos formas:

---

#### 📌 Forma 1: Lectura directa desde el archivo

Esta forma permite consultar directamente los archivos almacenados dentro del volumen indicando el formato del archivo en la sentencia `FROM`.

```sql
SELECT *
FROM formato_archivo.`path_volumen_formato_archivo`;
```

Es una alternativa rápida y sencilla cuando no necesitamos configurar opciones adicionales para la lectura del archivo.

#### 📌 Forma 2: Utilizando `read_files()`

Cuando necesitamos un mayor nivel de configuración durante la lectura, Spark SQL pone a disposición la función `read_files()`.

```sql
SELECT *
FROM read_files(
    'path_volumen_formato_archivo',
    format => 'formato_archivo',
    otras_opciones -- Dependerán del formato del archivo
);
```

A diferencia del primer enfoque, `read_files()` permite especificar diferentes opciones de lectura según el tipo de archivo que estemos procesando, proporcionando una mayor flexibilidad.

---

### 🔍 ¿Cuál es la diferencia?

Ambos enfoques permiten leer archivos almacenados en un Volumen mediante Spark SQL.

La principal diferencia radica en el nivel de configuración disponible durante la lectura:

* ✅ **Forma 1:** Ideal para lecturas simples donde no es necesario especificar opciones adicionales.

* ✅ **Forma 2 (`read_files()`):** Recomendada cuando el formato del archivo requiere parámetros adicionales de lectura o una configuración más personalizada.

---

### 📂 Uso de Wildcards

Ambas formas permiten utilizar **Wildcards (`*`)** para leer múltiples archivos de un mismo Volumen. Algunos escenarios comunes son:

* 📄 Leer todos los archivos del Volumen: 
`path_volumen_formato_archivo`

* 📄 Leer únicamente archivos de una determinada extensión: `path_volumen_formato_archivo/*.extension_archivo`

* 📄 Leer archivos que comiencen con un prefijo específico: `path_volumen_formato_archivo/prefijo-*.extension_archivo`

Gracias a los Wildcards es posible filtrar fácilmente qué archivos serán procesados, permitiendo adaptar la lectura según la organización existente dentro del Volumen.


In [0]:
### ============== CONFIGURACIÓN PREVIA PARA LECTURA CON SPARK SQL ============================= ###

### CREAMOS UN VOLUMEN PARA LOS EJEMPLOS DE LECTURA

catalog = "catalog_databricks_2026_de"
schema = "schema_databricks_2026_de"
volumen = "source_data"

# spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volumen}")
print("Volumen creado correctamente")


### CREAMOS CARPETAS DENTRO DEL VOLUMEN (CSV y JSON)

path_datasets_csv = f"/Volumes/{catalog}/{schema}/{volumen}/csv" 
path_datasets_json = f"/Volumes/{catalog}/{schema}/{volumen}/json"
path_datasets_binary = f"/Volumes/{catalog}/{schema}/{volumen}/binary"
path_datasets_parquet = f"/Volumes/{catalog}/{schema}/{volumen}/parquet"

dbutils.fs.mkdirs(path_datasets_csv) ## CSV
dbutils.fs.mkdirs(path_datasets_json) ## JSON
dbutils.fs.mkdirs(path_datasets_binary) ## BINARY
dbutils.fs.mkdirs(path_datasets_parquet) ## PARQUET
print("Carpetas CSV, JSON, BINARY y PARQUET creadas correctamente")


### CARGAMOS DATOS DE STORAGE CLOUD
dbutils.fs.cp(source="s3://bucket-brayan-datasets/csv_datasets/",dest=path_datasets_csv,recurse=True)
dbutils.fs.cp(source="s3://bucket-brayan-datasets/json_datasets/",dest=path_datasets_json,recurse=True)
dbutils.fs.cp(source="s3://bucket-brayan-datasets/binary_datasets/",dest=path_datasets_binary,recurse=True)
dbutils.fs.cp(source="s3://bucket-brayan-datasets/parquet_datasets/",dest=path_datasets_parquet,recurse=True)
print("Archivos copiados exitosamente")


#### ========= CSV =============

In [0]:
### ============== LECTURAS CON SPARK SQL ============================= ###

#### A). FORMA 1: LECTURA DIRECTA DESDE EL ARCHIVO (sin wildcard)
display(spark.sql("""
                  
                  SELECT *
                  FROM CSV.`/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/csv/sales_client_1.csv`

                  """))

#### B). FORMA 1: LECTURA DIRECTA DESDE EL ARCHIVO (con wildcard)
display(spark.sql("""
                  
                  SELECT *
                  FROM CSV.`/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/csv/sales_client_*.csv`

                  """))

#### C). FORMA 2: UTILIZANDO READ_FILES (sin wildcard)
display(spark.sql("""
                  
                  SELECT *
                  FROM read_files(
                    '/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/csv/sales_client_1.csv',
                    format => 'csv',
                    header => true,
                    delimiter => ',',
                    inferSchema => true
                  ) 

                  """))

#### D). FORMA 2: UTILIZANDO READ_FILES (con wildcard)
display(spark.sql("""
                  
                  SELECT *
                  FROM read_files(
                    '/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/csv/sales_client_*.csv',
                    format => 'csv',
                    header => true,
                    delimiter => ',',
                    inferSchema => true
                  ) 

                  """))

#### ========= JSON =============

In [0]:
### ============== LECTURAS CON SPARK SQL ============================= ###

"""
    💡 Los JSON pueden ser de una sola linea o tener múltiples líneas.
        En estos ejemplos, manejaremos ambos tipos.
"""

#### A). FORMA 1: LECTURA DIRECTA DESDE EL ARCHIVO (sin wildcard)
display(spark.sql("""
                  
                  SELECT *
                  FROM JSON.`/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/json/logs_oneline_1.json`

                  """))

#### B). FORMA 1: LECTURA DIRECTA DESDE EL ARCHIVO (con wildcard)
display(spark.sql("""
                  
                  SELECT *
                  FROM JSON.`/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/json/logs_oneline_*.json`

                  """))

#### C). FORMA 2: UTILIZANDO READ_FILES (sin wildcard)

json_schema = """
`timestamp` TIMESTAMP,
user_id STRING,
service STRING,
action STRING,
resource_id STRING,
status STRING,
response_time_ms INT,
ip_address STRING,
region STRING,
error_code STRING,
log_level STRING
"""

display(spark.sql(f"""
                  
                  SELECT *
                  FROM read_files(
                    '/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/json/logs_multiline_1.json',
                    format => 'json',
                    schema => '{json_schema}'
                  ) 

                  """))

#### D). FORMA 2: UTILIZANDO READ_FILES (con wildcard)

display(spark.sql(f"""
                  
                  SELECT *
                  FROM read_files(
                    '/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/json/logs_multiline_*.json',
                    format => 'json',
                    schema => '{json_schema}'
                  ) 

                  """))

#### ========= BINARIO =============

In [0]:
### ============== LECTURAS CON SPARK SQL ============================= ###

#### A). FORMA 1: LECTURA DIRECTA DESDE EL ARCHIVO (sin wildcard)
display(spark.sql("""
                  
                  SELECT path,length,modificationTime
                  FROM BINARYFILE.`/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/binary/streamingprojecta.png`

                  """))

#### B). FORMA 1: LECTURA DIRECTA DESDE EL ARCHIVO (con wildcard)
display(spark.sql("""
                  
                  SELECT path,length,modificationTime
                  FROM BINARYFILE.`/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/binary/Perspective-*.png`

                  """))

# #### C). FORMA 2: UTILIZANDO READ_FILES (sin wildcard)
display(spark.sql("""
                  
                  SELECT path,length,modificationTime
                  FROM read_files(
                    '/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/binary/streamingprojecta.png',
                    format => 'binaryFile'
                  ) 

                  """))

# #### D). FORMA 2: UTILIZANDO READ_FILES (con wildcard)
display(spark.sql("""
                  
                  SELECT path,length,modificationTime
                  FROM read_files(
                    '/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/binary/Perspective-*.png',
                    format => 'binaryFile'
                  ) 

                  """))

#### ========= PARQUET (FORMATO COLUMNAR OPTIMIZADO) =============

In [0]:
### ============== LECTURAS CON SPARK SQL ============================= ###

#### A). FORMA 1: LECTURA DIRECTA DESDE EL ARCHIVO (sin wildcard)
display(spark.sql("""
                  
                  SELECT *
                  FROM PARQUET.`/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/parquet/infraestructura_costos.parquet`

                  """))

# #### B). FORMA 1: LECTURA DIRECTA DESDE EL ARCHIVO (con wildcard)
display(spark.sql("""
                  
                  SELECT *
                  FROM PARQUET.`/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/parquet/logs_actividad_*.parquet`

                  """))

# # #### C). FORMA 2: UTILIZANDO READ_FILES (sin wildcard)
display(spark.sql("""
                  
                  SELECT *
                  FROM read_files(
                    '/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/parquet/infraestructura_costos.parquet',
                    format => 'parquet'
                  ) 

                  """))

# # #### D). FORMA 2: UTILIZANDO READ_FILES (con wildcard)
display(spark.sql("""
                  
                  SELECT *
                  FROM read_files(
                    '/Volumes/catalog_databricks_2026_de/schema_databricks_2026_de/source_data/parquet/logs_actividad_*.parquet',
                    format => 'parquet'
                  ) 

                  """))